In [10]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = SparkConf()
conf.set('spark.app.name', 'Spark Accumulator')
conf.set('spark.master', 'local[*]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [11]:
data = {
    ('California', 'Sunnyvale', 9511),
    ('California', 'Mountain View', 94111),
    ('California', 'Cupertino', 94123),
    ('California', 'San Jose', 951),
}

df = spark.createDataFrame(data).toDF('state', 'city', 'zipcode')

df.show()

+----------+-------------+-------+
|     state|         city|zipcode|
+----------+-------------+-------+
|California|    Cupertino|  94123|
|California|    Sunnyvale|   9511|
|California|     San Jose|    951|
|California|Mountain View|  94111|
+----------+-------------+-------+



In [15]:
bad_zipcodes = spark.sparkContext.accumulator(0)

In [16]:
import pyspark.sql.functions as f
from pyspark.sql.types import IntegerType

def handle_bad_zipcode(c: int) -> int:
    if len(str(c)) != 5:
        bad_zipcodes.add(1)
        return None
    return c

handle_bad_zipcode_udf = f.udf(handle_bad_zipcode, IntegerType())

C:\Spark\spark-4.1.1-bin-hadoop3\python\lib\pyspark.zip\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 


In [17]:
df.withColumn('correct_zipcode', handle_bad_zipcode_udf(f.col('zipcode'))).show()

+----------+-------------+-------+---------------+
|     state|         city|zipcode|correct_zipcode|
+----------+-------------+-------+---------------+
|California|    Cupertino|  94123|          94123|
|California|    Sunnyvale|   9511|           NULL|
|California|     San Jose|    951|           NULL|
|California|Mountain View|  94111|          94111|
+----------+-------------+-------+---------------+



In [18]:
bad_zipcodes

Accumulator<id=7, value=2>

In [19]:
bad_zipcodes.value

2

**DataFrame Foreach**

In [ ]:
zipcode = spark.sparkContext.accumulator(0)

def add_accumulator(row):
    if len(str(row['zipcode'])) != 5:
        zipcode.add(1)

df.foreach(add_accumulator)